# Run Overview

In [34]:
# %% [markdown]
# # AI Quiz Competition Dashboard
# Modello: `qwen2.5:7b-instruct` — multi-agent (orchestrator + math + knowledge)

# %% — Installazione dipendenze (eseguire solo la prima volta)
# !pip install plotly pandas

# %% — Import e dati
import json, glob, os
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

JSON_DIR = "../log/fullrun_202605311451_qwen2_5_7b-instruct"

run_info_path = os.path.join(JSON_DIR, ".run_info.json")
with open(run_info_path) as f:
    run_info = json.load(f)

comp_files = sorted(glob.glob(os.path.join(JSON_DIR, "comp_*.json")))
competitions_raw = []
for fp in comp_files:
    with open(fp, "r", encoding="utf-8") as f:
        competitions_raw.append(json.load(f))

# %% — DataFrame competizioni
comp_rows = []
for cr in competitions_raw:
    s = cr["summary"]
    comp_rows.append({
        "id":         cr["competition_id"],
        "name":       cr["competition_name"],
        "accuracy":   round(s["accuracy"] * 100, 2),
        "avg_level":  s["avg_level"],
        "max_level":  s["max_level"],
        "avg_earned": s["avg_earned"],
        "max_earned": s["max_earned"],
        "questions":  s["questions_answered"],
        "timeouts":   s["timeout_count"],
        "avg_time_q": round(s["avg_time_per_question"], 2),
    })
df_comp = pd.DataFrame(comp_rows)

# %% — DataFrame tentativi
attempt_rows = []
for cr in competitions_raw:
    for att in cr["attempts"]:
        attempt_rows.append({
            "comp_id":   cr["competition_id"],
            "comp_name": cr["competition_name"],
            "attempt":   att["attempt"],
            "earned":    att["earned"],
            "level":     att["final_level"],
        })
df_att = pd.DataFrame(attempt_rows)

# %% — DataFrame domande
q_rows = []
for cr in competitions_raw:
    for att in cr["attempts"]:
        for q in att["questions"]:
            q_rows.append({
                "comp_id":   cr["competition_id"],
                "comp_name": cr["competition_name"],
                "attempt":   att["attempt"],
                "level":     q["level"],
                "correct":   q.get("correct"),
                "timed_out": q["timed_out"],
                "time_sec":  round(q["time"], 2),
            })
df_q = pd.DataFrame(q_rows)

# %% — Palette colori
COLORS = {
    "Entertainment":                "#b57514",
    "Ancient History and Politics": "#185fa5",
    "Science and Nature":           "#0f6e56",
    "Maths":                        "#533ab7",
    "Philosophy and Psychology":    "#953cdd",
    "News":                         "#a32d2d",
}
comp_color_list = [COLORS.get(n, "#888") for n in df_comp["name"]]

# %% — Griglia 2×2: tutti e 4 i grafici principali
def close(lst): return lst + [lst[0]]

names        = df_comp["name"].tolist()
acc          = df_comp["accuracy"].tolist()
lvl_max_norm = (df_comp["max_level"] / 15 * 100).round(1).tolist()
df_timeouts  = df_q[df_q["timed_out"] == True].copy()

fig = make_subplots(
    rows=2, cols=2,
    specs=[
        [{"type": "xy"},    {"type": "xy"}],
        [{"type": "polar"}, {"type": "xy"}],
    ],
    subplot_titles=[
        "Accuratezza per categoria (%)",
        "Livello medio vs massimo raggiunto",
        "Radar: accuratezza (viola) vs livello max (grigio)",
        "Tempo medio per domanda e timeout (s)",
    ],
    vertical_spacing=0.30,
    horizontal_spacing=0.15,
)

# ── 1. Accuratezza — una traccia per categoria (colori distinti)
for _, row in df_comp.iterrows():
    fig.add_trace(
        go.Bar(
            x=[row["name"]], y=[row["accuracy"]],
            marker_color=COLORS.get(row["name"], "#888"),
            text=[f"{row['accuracy']:.1f}%"], textposition="outside",
            showlegend=False,
        ),
        row=1, col=1,
    )

# ── 2. Livello medio vs massimo
fig.add_trace(
    go.Bar(x=names, y=df_comp["max_level"],
           name="Livello max", marker_color="lightgray", showlegend=True),
    row=1, col=2,
)
fig.add_trace(
    go.Bar(x=names, y=df_comp["avg_level"],
           name="Livello medio", marker_color=comp_color_list, showlegend=True),
    row=1, col=2,
)

# ── 3. Radar
fig.add_trace(
    go.Scatterpolar(
        r=close(lvl_max_norm), theta=close(names),
        fill="toself", fillcolor="rgba(150,150,150,.18)",
        line=dict(color="rgba(130,130,130,.6)", width=1.5, dash="dot"),
        name="Livello max (norm.)",
    ),
    row=2, col=1,
)
fig.add_trace(
    go.Scatterpolar(
        r=close(acc), theta=close(names),
        fill="toself", fillcolor="rgba(149,60,221,.15)",
        line=dict(color="#953cdd", width=2),
        name="Accuratezza (%)",
    ),
    row=2, col=1,
)

# ── 4. Tempo medio + punti timeout
fig.add_trace(
    go.Bar(
        x=names, y=df_comp["avg_time_q"],
        name="Tempo medio (s)", marker_color=comp_color_list,
        text=df_comp["avg_time_q"].apply(lambda v: f"{v:.1f}s"),
        textposition="outside", showlegend=False,
    ),
    row=2, col=2,
)
fig.add_trace(
    go.Scatter(
        x=df_timeouts["comp_name"], y=df_timeouts["time_sec"],
        name="Timeout",
        mode="markers",
        marker=dict(symbol="diamond", size=10, color="#a32d2d",
                    line=dict(width=1.5, color="white")),
    ),
    row=2, col=2,
)

# ── Layout globale
fig.update_layout(
    height=820,
    title_text=(
        f"AI Quiz Dashboard — {run_info['model_name']}"
        f"  |  Accuratezza globale: {run_info['overall_accuracy']*100:.1f}%"
    ),
    title_font_size=14,
    plot_bgcolor="white",
    paper_bgcolor="white",
    barmode="overlay",
    legend=dict(orientation="h", y=-0.04, x=0, font_size=11),
)

# ── Assi per subplot
fig.update_yaxes(range=[0, 110], row=1, col=1)
fig.update_yaxes(range=[0, 15],  title_text="Livello", row=1, col=2)
fig.update_yaxes(
    range=[0, df_timeouts["time_sec"].max() * 1.25 if len(df_timeouts) else 40],
    title_text="Secondi", gridcolor="#eee", row=2, col=2,
)
fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 100])))

fig.show()

# %% — Statistiche riepilogative
print(f"\n{'='*55}")
print(f"  Modello:             {run_info['model_name']}")
print(f"  Accuratezza globale: {run_info['overall_accuracy']*100:.2f}%")
print(f"  Competizioni:        {run_info['total_competitions']}")
print(f"  Run per categoria:   {run_info['num_runs_per_competition']}")
print(f"  Domande totali:      {df_q.shape[0]}")
print(f"  Timeout totali:      {df_q['timed_out'].sum()}")
print(f"  Tempo medio/dom.:    {df_q['time_sec'].mean():.1f}s")
print(f"{'='*55}")
print(df_comp[["name","accuracy","avg_level","avg_earned","max_earned","timeouts"]]
      .rename(columns={"name":"Categoria","accuracy":"Acc (%)","avg_level":"Lvl medio",
                       "avg_earned":"Avg €","max_earned":"Max €","timeouts":"Timeout"})
      .to_string(index=False))


  Modello:             qwen2.5:7b-instruct
  Accuratezza globale: 37.56%
  Competizioni:        6
  Run per categoria:   5
  Domande totali:      196
  Timeout totali:      5
  Tempo medio/dom.:    16.1s
                   Categoria  Acc (%)  Lvl medio    Avg €   Max €  Timeout
               Entertainment    17.33        2.6    300.0     500        0
Ancient History and Politics    36.00        5.4 103220.0  512000        0
          Science and Nature    58.67        8.8  42600.0  128000        0
                       Maths    17.33        2.6   3280.0   16000        1
   Philosophy and Psychology    82.67       12.4 621600.0 1024000        1
                        News    13.33        2.0    240.0     500        3


In [52]:
# %% — Dashboard di Confronto: Bm25 vs sBert (Categoria: News)
import json
import glob
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Trova tutti i file JSON pertinenti nella cartella. 
percorsi_file = glob.glob("../log/comp_5_News_*.json")

dati_estratti = []

# 2. Cicla attraverso ogni file trovato
for percorso in percorsi_file:
    with open(percorso, 'r', encoding='utf-8') as f:
        dati_json = json.load(f)
        
        # 3. Determina la strategia guardando il nome del file
        nome_file_lower = percorso.lower()
        if "bm25" in nome_file_lower or "basic" in nome_file_lower: # basic lo consideriamo bm25 in base ai tuoi vecchi file
            strategia = "Bm25"
        elif "sbert" in nome_file_lower:
            strategia = "sBert"
        else:
            continue
            
        # 4. Estrai le metriche dal dizionario "summary"
        summary = dati_json.get("summary", {})
        
        # 5. Salva i dati in un formato compatibile con il nostro grafico
        dati_estratti.append({
            "strategia": strategia,
            "accuratezza": summary.get("accuracy", 0) * 100,
            "lvl_max": summary.get("max_level", 0),
            "guadagno": summary.get("avg_earned", 0),
            "tempo": summary.get("avg_time_per_question", 0)
        })

# 6. Crea il DataFrame e aggrega
df_ab = pd.DataFrame(dati_estratti)

if df_ab.empty:
    print("Nessun dato trovato. Controlla il percorso dei file in glob.glob()")
else:
    # Aggregazione dei dati (Media per Accuratezza, Guadagno e Tempo. Massimo per il Livello)
    df_agg = df_ab.groupby("strategia").agg({
        "accuratezza": "mean",
        "lvl_max": "max",
        "guadagno": "mean",
        "tempo": "mean"
    }).reset_index()

    # 7. Impostazione colori
    color_map = {"Bm25": "#185fa5", "sBert": "#953cdd"}
    colori_barre = [color_map.get(strat, "#888") for strat in df_agg["strategia"]]

    # 8. Creazione della griglia 1x4 SENZA subplot_titles
    fig_ab = make_subplots(
        rows=1, cols=4,
        horizontal_spacing=0.08
    )

    # ── Colonna 1: Accuratezza ──
    fig_ab.add_trace(
        go.Bar(
            x=df_agg["strategia"], y=df_agg["accuratezza"],
            marker_color=colori_barre,
            text=df_agg["accuratezza"].apply(lambda v: f"{v:.1f}%"),
            textposition="outside", showlegend=False
        ), row=1, col=1
    )

    # ── Colonna 2: Livello Max ──
    fig_ab.add_trace(
        go.Bar(
            x=df_agg["strategia"], y=df_agg["lvl_max"],
            marker_color=colori_barre,
            text=df_agg["lvl_max"].apply(lambda v: f"Lvl {v}"),
            textposition="outside", showlegend=False
        ), row=1, col=2
    )

    # ── Colonna 3: Guadagno Medio ──
    fig_ab.add_trace(
        go.Bar(
            x=df_agg["strategia"], y=df_agg["guadagno"],
            marker_color=colori_barre,
            text=df_agg["guadagno"].apply(lambda v: f"€ {v:,.0f}"),
            textposition="outside", showlegend=False
        ), row=1, col=3
    )

    # ── Colonna 4: Tempo Medio ──
    fig_ab.add_trace(
        go.Bar(
            x=df_agg["strategia"], y=df_agg["tempo"],
            marker_color=colori_barre,
            text=df_agg["tempo"].apply(lambda v: f"{v:.1f}s"),
            textposition="outside", showlegend=False
        ), row=1, col=4
    )

    # 9. Formattazione Layout Globale
    fig_ab.update_layout(
        height=450,
        title_text="A/B Testing: Strategia Bm25 vs Strategia sBert (Categoria: News)",
        title_font_size=16,
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(b=80, t=60) # Spazio inferiore per i titoli
    )

    # 10. Assegnazione dei titoli SOTTO i grafici
    fig_ab.update_xaxes(title_text="<b>Accuratezza Media (%)</b>", title_font=dict(size=13), row=1, col=1)
    fig_ab.update_xaxes(title_text="<b>Livello Max Raggiunto</b>", title_font=dict(size=13), row=1, col=2)
    fig_ab.update_xaxes(title_text="<b>Guadagno Medio (€)</b>", title_font=dict(size=13), row=1, col=3)
    fig_ab.update_xaxes(title_text="<b>Tempo per Domanda (s)</b>", title_font=dict(size=13), row=1, col=4)

    # 11. Aggiornamento assi Y (tutti sulla riga 1)
    fig_ab.update_yaxes(range=[0, max(df_agg["accuratezza"])*1.3 if max(df_agg["accuratezza"]) > 0 else 100], row=1, col=1)
    fig_ab.update_yaxes(range=[0, 15], row=1, col=2)
    fig_ab.update_yaxes(range=[0, max(df_agg["guadagno"])*1.2 if max(df_agg["guadagno"]) > 0 else 1000], row=1, col=3)
    fig_ab.update_yaxes(range=[0, max(df_agg["tempo"])*1.3 if max(df_agg["tempo"]) > 0 else 20], row=1, col=4)

    fig_ab.update_xaxes(gridcolor="#eee", zerolinecolor="#ccc")
    fig_ab.update_yaxes(gridcolor="#eee", zerolinecolor="#ccc")

    fig_ab.show()

In [51]:

# %% — Dashboard di Confronto: Basic vs Coding (Categoria: Maths)
import json
import glob
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Trova automaticamente i file JSON pertinenti alla categoria Maths
percorsi_file = glob.glob("../log/comp_3_Maths_*.json")

dati_estratti = []

# 2. Cicla attraverso ogni file trovato per estrarre le metriche
for percorso in percorsi_file:
    with open(percorso, 'r', encoding='utf-8') as f:
        dati_json = json.load(f)
        
        # Determina la strategia dal nome del file
        nome_file_lower = percorso.lower()
        if "basic" in nome_file_lower:
            strategia = "Basic"
        elif "coding" in nome_file_lower:
            strategia = "Coding"
        else:
            continue # Salta file non pertinenti
            
        summary = dati_json.get("summary", {})
        
        dati_estratti.append({
            "strategia": strategia,
            "accuratezza": summary.get("accuracy", 0) * 100,
            "lvl_max": summary.get("max_level", 0),
            "guadagno": summary.get("avg_earned", 0),
            "tempo": summary.get("avg_time_per_question", 0)
        })

# 3. Creazione del DataFrame e aggregazione
df_ab = pd.DataFrame(dati_estratti)

if df_ab.empty:
    print("Nessun dato trovato. Controlla il percorso dei file in glob.glob()")
else:
    # Raggruppiamo calcolando le medie (e il massimo per il livello)
    df_agg = df_ab.groupby("strategia").agg({
        "accuratezza": "mean",
        "lvl_max": "max",
        "guadagno": "mean",
        "tempo": "mean"
    }).reset_index()

    # 4. Impostazione colori (Viola per Maths/Basic, Arancione per Coding)
    color_map = {"Basic": "#533ab7", "Coding": "#e67e22"}
    colori_barre = [color_map.get(strat, "#888") for strat in df_agg["strategia"]]

    # 5. Creazione della griglia 1x4 (SENZA subplot_titles)
    fig_ab = make_subplots(
        rows=1, cols=4,
        horizontal_spacing=0.08
    )

    # ── Colonna 1: Accuratezza ──
    fig_ab.add_trace(
        go.Bar(
            x=df_agg["strategia"], y=df_agg["accuratezza"],
            marker_color=colori_barre,
            text=df_agg["accuratezza"].apply(lambda v: f"{v:.1f}%"),
            textposition="outside", showlegend=False
        ), row=1, col=1
    )

    # ── Colonna 2: Livello Max ──
    fig_ab.add_trace(
        go.Bar(
            x=df_agg["strategia"], y=df_agg["lvl_max"],
            marker_color=colori_barre,
            text=df_agg["lvl_max"].apply(lambda v: f"Lvl {v}"),
            textposition="outside", showlegend=False
        ), row=1, col=2
    )

    # ── Colonna 3: Guadagno Medio ──
    fig_ab.add_trace(
        go.Bar(
            x=df_agg["strategia"], y=df_agg["guadagno"],
            marker_color=colori_barre,
            text=df_agg["guadagno"].apply(lambda v: f"€ {v:,.0f}"),
            textposition="outside", showlegend=False
        ), row=1, col=3
    )

    # ── Colonna 4: Tempo Medio ──
    fig_ab.add_trace(
        go.Bar(
            x=df_agg["strategia"], y=df_agg["tempo"],
            marker_color=colori_barre,
            text=df_agg["tempo"].apply(lambda v: f"{v:.1f}s"),
            textposition="outside", showlegend=False
        ), row=1, col=4
    )

    # 6. Formattazione Layout Globale
    fig_ab.update_layout(
        height=450, # Leggermente più alto per far spazio ai titoli in basso
        title_text="A/B Testing: Strategia Basic vs Strategia Coding (Categoria: Maths)",
        title_font_size=16,
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(b=80, t=60) # Aumentato il margine inferiore (b) per non tagliare i titoli
    )

    # 7. Assegnazione dei titoli SOTTO i grafici usando gli assi X
    fig_ab.update_xaxes(title_text="<b>Accuratezza Media (%)</b>", title_font=dict(size=13), row=1, col=1)
    fig_ab.update_xaxes(title_text="<b>Livello Max Raggiunto</b>", title_font=dict(size=13), row=1, col=2)
    fig_ab.update_xaxes(title_text="<b>Guadagno Medio (€)</b>", title_font=dict(size=13), row=1, col=3)
    fig_ab.update_xaxes(title_text="<b>Tempo per Domanda (s)</b>", title_font=dict(size=13), row=1, col=4)

    # Aggiornamento limiti assi Y e griglie
    fig_ab.update_yaxes(range=[0, max(df_agg["accuratezza"])*1.3 if max(df_agg["accuratezza"]) > 0 else 100], row=1, col=1)
    fig_ab.update_yaxes(range=[0, 15], row=1, col=2)
    fig_ab.update_yaxes(range=[0, max(df_agg["guadagno"])*1.2 if max(df_agg["guadagno"]) > 0 else 1000], row=1, col=3)
    fig_ab.update_yaxes(range=[0, max(df_agg["tempo"])*1.3 if max(df_agg["tempo"]) > 0 else 20], row=1, col=4)

    fig_ab.update_xaxes(gridcolor="#eee", zerolinecolor="#ccc")
    fig_ab.update_yaxes(gridcolor="#eee", zerolinecolor="#ccc")
    
    # Mostra il grafico
    fig_ab.show()